### HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. Language mismatch between query and documents
3.You want to retrieve based on answer content, not question words

In [1]:
from dotenv import load_dotenv
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.vectorstores import FAISS,Chroma
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
import warnings

warnings.filterwarnings("ignore")

In [2]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device":"cpu"}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs={**model_kwargs,"local_files_only":True}
)

In [3]:
load = load_dotenv()
llm = ChatGroq(model_name="openai/gpt-oss-20b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025C10BA49E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025C10B7C560>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
loader = WikipediaLoader(query="steve jobs", load_max_docs = 5)
documents = loader.load()

text_splitter =  RecursiveCharacterTextSplitter(
    chunk_size = 300, chunk_overlap = 50
)
docs = text_splitter.split_documents(documents=documents)
docs

[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Computer Company (now Apple Inc.) in 1976 with his early business partner Steve Wozniak. After the company\'s board of directors fired him in 1985, he founded NeXT in the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop

In [7]:
vectorstore = FAISS.from_documents(docs,embeddings)
vectorstore

In [9]:
#creating vectorStore
db = Chroma.from_documents(
    documents = docs,
    embedding = embeddings,
    persist_directory = "output/steve_jobs_for_hype.db"
)
#creating the retriever
retriever = db.as_retriever(search_kwargs={"k":5})

In [13]:
#generating a prompt for HYDE
from langchain_core.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""
    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    message = chat_prompt.format_prompt(query=query).to_messages()
    print(message)
    response = llm.invoke(message)
    hypo_doc = response.content
    return hypo_doc

In [15]:
query = "When was Steve Jobs fired from Apple?"
print(get_hyde_doc(query=query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
**When was Steve Jobs fired from Apple?**

Steve Jobs was effectively forced out of Apple on **June 13, 1985**.  That date marks the board‑room decision that ended his tenure as head of the Macintosh division and, ultimately, his position within the company he co‑founded a decade earlier.

---

### The Road to the Board Meeting

| Year | Key Event | Impact on Jobs |
|------|-----------|----------------|
| **1976** | Jobs, Steve Wozniak, and Ronald Wayne form Apple Computer, Inc. | Jobs becomes chief visionary and product architect. |
| **1980** | Apple goes public (IPO). | Apple’s stock surges, giving Jobs significant influence. |
| **1983** | John Sculley is hired as CEO. | Sculley’s corporate experience clashes with Jobs’ more radical style. |
| **1984** | M

In [17]:
matched_docs = retriever.invoke(get_hyde_doc(query))
print(matched_docs)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Computer Company (now Apple Inc.) in 1976 with his early business partner Steve Wozniak. After the company\'s board of directors fired him in 1985, he founded NeXT in the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended R

### langchain - Hypothetical Document Embedder

In [25]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain_classic.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50);
chunks = splitter.split_documents(docs)

In [26]:
#LLM 
base_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

In [27]:
#step 3 HYDE embedder
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key = "web_search"
)

In [29]:
#step 4 store doc in vectorstore Chromadb
vector_store = Chroma.from_documents(
    documents= chunks,
    embedding = hyde_embedding_function,
    persist_directory = "output/langchain"
)

In [31]:
# step 5 rag answer generation prompt
rag_prompt = PromptTemplate.from_template(
    """
Use the context below to answer the question.

Context:
{context}

Question: {input}
"""
)
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [34]:
## RAG pipeline
def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query,k=4)
    print(matched_docs)
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    return response

In [35]:
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print(answer)

[Document(id='e05edc79-edc1-4b98-ac80-e54f5adf5374', metadata={'title': 'Eve Jobs', 'summary': 'Eve Jobs (born July 9, 1998) is an American fashion model. She is the daughter of the late Apple Inc. co-founder Steve Jobs and businesswoman Laurene Powell Jobs.', 'source': 'https://en.wikipedia.org/wiki/Eve_Jobs'}, page_content='== Early life =='), Document(id='0a903af8-420c-4be5-9e07-1f0f4b4b7256', metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Computer Company (now Apple Inc.) in 1976 with his early business partner Steve Wozniak. After the company\'s board of directors fired him in 1985, he founded NeXT in the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creatio